# **Ruta Purista · Track 2: Aumentación de datos con ModernLab**

Este notebook es la ruta recomendada para Track 2 después de tokenización y preprocesamiento. ModernLab es el camino principal; el notebook legacy `05_Data_Augmentation_Using_NLPaug.ipynb` queda solo como referencia histórica y comparación.

**Objetivo**

- Practicar técnicas de data augmentation para texto.
- Entender cómo cada perturbación cambia el input.
- Aprender cuándo la augmentación mejora la robustez y cuándo puede dañar la señal.

**Qué vas a aprender**

- Ruido a nivel de caracteres.
- Reemplazos y transformaciones a nivel de palabras.
- Una introducción al reemplazo contextual con Transformers.

**Qué vas a obtener**

- Ejemplos listos para inspeccionar.
- Una intuición práctica de qué tipo de ruido ayuda.
- Un criterio para no sobre-augmentar ni cambiar el significado.

**Por qué va después de preprocesamiento/tokenización**

- La augmentación funciona mejor cuando el texto base ya está limpio y consistente.
- Si se aplica demasiado pronto, amplifica ruido, ambigüedad y errores del corpus.
- Primero normalizamos; después generamos variaciones controladas.

**Cuándo ayuda**

- OCR o typos.
- Variantes ortográficas.
- Sinonimia moderada.
- Robustez ante borrado o intercambio de palabras.
- Reescritura contextual.

**Cuándo puede perjudicar**

- Cuando altera demasiado el significado.
- Cuando introduce ruido artificial que no se parece al mundo real.
- Cuando cambia entidades, negación o términos críticos del dominio.

> Inspirado y curado a partir de *Practical Natural Language Processing* (O'Reilly), <https://www.practicalnlp.ai/> y <https://github.com/practical-nlp/practical-nlp-code/blob/master/Ch2/>. Material adaptado para uso educativo; no representa afiliación oficial.


In [ ]:
# ============================================================
# NLP Data Augmentation Modern Lab
# Ruta preferida de Track 2: ModernLab
# Legacy NLPaug notebook = referencia histórica/comparación
# ============================================================

# Limpieza opcional para sesiones desechables de Colab.
# Solo descomenta esto si estás en una sesión descartable o si encuentras conflictos de versiones.
# En entornos locales persistentes, evita borrar paquetes por defecto.
# !pip uninstall -y nlpaug transformers tokenizers textattack flair huggingface-hub

# Debido a conflictos persistentes de dependencias y a problemas de compilación con `tokenizers < 0.14`
# (requerido por flair 0.15.1, que a su vez usa TextAttack 0.3.10) en Colab con Python 3.12,
# TextAttack 0.3.10 no se puede instalar de forma confiable en este entorno.
# Del mismo modo, los modelos contextuales de nlpaug 1.1.11 requieren versiones antiguas de
# transformers/tokenizers que no son compatibles con la versión actual de Python 3.12 en Colab.

# Estrategia: instalar nlpaug para sus augmenters no contextuales.
# La augmentación contextual con nlpaug 1.1.11 no es viable en este entorno.

# 1. Instalar nlpaug 1.1.11 para sus augmenters no contextuales y dependencias generales.
!pip install -q nlpaug==1.1.11 pandas scikit-learn matplotlib

# 2. Instalar aparte las versiones estables de transformers, tokenizers y huggingface-hub.
#    Estarán disponibles para otros usos, pero los modelos contextuales de nlpaug
#    no podrán usarlas por incompatibilidad de API.
!pip install -q transformers tokenizers huggingface-hub

# Verificar versiones instaladas con fines de depuración.
!pip show nlpaug transformers tokenizers textattack flair huggingface-hub

import random
import pandas as pd
import matplotlib.pyplot as plt

import nltk
nltk.download("wordnet")
nltk.download("omw-1.4")
nltk.download("averaged_perceptron_tagger_eng")

import nlpaug.augmenter.char as nac
import nlpaug.augmenter.word as naw
from nlpaug.util import Action


**1. Texto base**

In [ ]:
text = "The quick brown fox jumps over the lazy dog."

print("Original text:")
print(text)

**2. Character-level augmentation**

Este bloque sirve para enseñar errores tipo OCR, teclado, ruido de entrada y robustez de modelos.

In [ ]:
# OCR-like augmentation
ocr_aug = nac.OcrAug()

ocr_examples = ocr_aug.augment(text, n=5)

print("OCR Augmentation:")
for example in ocr_examples:
    print("-", example)

In [ ]:
# Keyboard typo augmentation
keyboard_aug = nac.KeyboardAug()

keyboard_examples = keyboard_aug.augment(text, n=5)

print("Keyboard Augmentation:")
for example in keyboard_examples:
    print("-", example)

**3. Word-level augmentation con sinónimos**

In [ ]:
# Synonym augmentation using WordNet
synonym_aug = naw.SynonymAug(aug_src="wordnet")

synonym_examples = synonym_aug.augment(text, n=5)

print("Synonym Augmentation:")
for example in synonym_examples:
    print("-", example)

**4. Random word augmentation**

In [ ]:
# Random delete
delete_aug = naw.RandomWordAug(action="delete")

delete_examples = delete_aug.augment(text, n=5)

print("Random Delete Augmentation:")
for example in delete_examples:
    print("-", example)

In [ ]:
# Random swap
swap_aug = naw.RandomWordAug(action="swap")

swap_examples = swap_aug.augment(text, n=5)

print("Random Swap Augmentation:")
for example in swap_examples:
    print("-", example)

**5. Contextual augmentation con Transformers**

Esta parte usa un modelo contextual. En la primera ejecución puede descargar `distilbert-base-uncased`, necesita conexión a internet y tarda un poco más en arrancar.

Aquí conectamos las reglas de augmentación con un modelo que sí mira el contexto completo de la frase.


In [ ]:
from transformers import pipeline
import random

# `distilbert-base-uncased` se descarga la primera vez.
# Esta sección necesita internet y puede tardar más en el arranque inicial.
fill_mask = pipeline("fill-mask", model="distilbert-base-uncased")

# Original text (assuming 'text' variable is still defined from previous cells)
print(f"Original text: {text}\n")

# Function to perform one contextual augmentation with top-k or top-p sampling
def augment_text_with_transformer(original_text, num_augmentations=1, top_k=None, top_p=None):
    augmented_texts = []
    words = original_text.split()

    for _ in range(num_augmentations):
        # Choose a random word to mask
        if not words:
            augmented_texts.append(original_text) # Handle empty string case
            continue

        masked_word_index = random.randint(0, len(words) - 1)
        original_word = words[masked_word_index]

        # Create the masked sentence
        masked_sentence_parts = words[:masked_word_index] + ['[MASK]'] + words[masked_word_index+1:]
        masked_sentence = " ".join(masked_sentence_parts)

        # Get predictions for the masked token
        predictions = fill_mask(masked_sentence)

        predicted_token = original_word # Fallback to original word if no predictions are valid

        if predictions and len(predictions) > 0:
            # Sort predictions by score in descending order
            sorted_predictions = sorted(predictions, key=lambda x: x['score'], reverse=True)

            if top_p is not None and top_p > 0.0 and top_p < 1.0:
                # Apply top-p (nucleus) sampling
                cumulative_probability = 0.0
                filtered_predictions = []
                for pred in sorted_predictions:
                    cumulative_probability += pred['score']
                    filtered_predictions.append(pred)
                    if cumulative_probability >= top_p:
                        break

                if filtered_predictions: # Ensure there are predictions after filtering
                    chosen_prediction = random.choice(filtered_predictions)
                    predicted_token = chosen_prediction['token_str']
                else:
                    # Fallback to top-1 if top-p filtering results in empty set
                    predicted_token = sorted_predictions[0]['token_str']
            elif top_k is not None and top_k > 0:
                # Apply top-k sampling
                effective_top_k = min(top_k, len(sorted_predictions))
                chosen_prediction = random.choice(sorted_predictions[:effective_top_k])
                predicted_token = chosen_prediction['token_str']
            else:
                # Default to top-1 if neither top_k nor top_p are effectively set
                predicted_token = sorted_predictions[0]['token_str']

        # Replace the mask with the predicted token
        augmented_sentence_parts = words[:masked_word_index] + [predicted_token] + words[masked_word_index+1:]
        augmented_text = " ".join(augmented_sentence_parts)
        augmented_texts.append(augmented_text)

    return augmented_texts

# Generate 5 augmented examples using top-p=0.9 sampling
contextual_transformer_examples_topp = augment_text_with_transformer(text, num_augmentations=5, top_p=0.9)

print("Contextual Augmentation (via Transformers Pipeline with Top-P=0.9):")
for example_topp in contextual_transformer_examples_topp:
    print("-", example_topp)


# **Entrenar con variaciones: preparar un modelo para texto imperfecto del mundo real**

El texto real casi nunca llega limpio. Hay OCR defectuoso, teclas mal pulsadas, palabras omitidas, sinónimos inesperados y pequeñas reescrituras que cambian la superficie sin cambiar demasiado el sentido. La data augmentation sirve para preparar al modelo para ese ruido sin perder robustez.

## Qué hicimos

- **Ruido de caracteres**: simulamos errores de OCR y teclado.
- **Ruido de palabras**: borrado e intercambio aleatorio.
- **Sustitución por sinónimos**: variaciones leves con diccionario.
- **Reemplazo contextual**: una palabra se enmascara y un Transformer propone alternativas coherentes.

## Por qué importa

- Mejora la robustez ante typos, OCR y pequeñas variaciones.
- Ayuda a generalizar cuando hay poca data.
- Obliga al modelo a mirar más allá de una sola forma de escribir una idea.

## Cuidado

La augmentación también puede perjudicar si:

- rompe entidades o negaciones,
- cambia demasiado el significado,
- introduce ruido que no existe en tus datos reales.

En resumen: augmentar no es solo “hacer más datos”; es preparar el entrenamiento para el texto imperfecto del mundo real.
